# 00 Load Library
---


In [ ]:
import json
from pathlib import Path
import pandas as pd


# 01 Inisialisasi Path dan Konfigurasi
---
## Konfigurasi Direktori Kerja
---
### Penentuan Lokasi File Raw dan Output
---
Mendefinisikan path data mentah OpenStreetMap dan folder penyimpanan data bersih.


In [ ]:
BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == 'data_cleaning' else Path.cwd()
RAW_OSM_DIR = BASE_DIR / 'data' / 'raw' / 'osm'
OUTPUT_DIR = BASE_DIR / 'data' / 'cleaned'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# 02 Pembacaan Data Mentah OSM
---
## Ekstraksi File JSON
---
### Parsing Koordinat dan Kategori Usaha
---
Membaca seluruh berkas JSON hasil kueri Overpass API yang mencakup 5 sektor pada 5 kabupaten/kota di DIY.


In [ ]:
raw_records = []
for json_file in sorted(RAW_OSM_DIR.glob('*.json')):
    parts = json_file.stem.split('_')
    if len(parts) >= 2:
        wilayah = '_'.join(parts[:-1])
        sektor = parts[-1]
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for el in data.get('elements', []):
                lat = el.get('lat', el.get('center', {}).get('lat'))
                lon = el.get('lon', el.get('center', {}).get('lon'))
                name = el.get('tags', {}).get('name', 'Tanpa Nama')
                raw_records.append({
                    'osm_id': el.get('id'),
                    'name': name,
                    'sektor': sektor,
                    'wilayah': wilayah,
                    'latitude': lat,
                    'longitude': lon
                })
df_osm_raw = pd.DataFrame(raw_records)
print('Total record mentah diekstraksi:', len(df_osm_raw))


In [ ]:
df_osm_raw.head()


# 03 Pembersihan Data dan Validasi Geospasial
---
## Filter Batas Geografis
---
### Validasi Koordinat Bounding Box D.I. Yogyakarta
---
Menghapus baris dengan koordinat null dan memvalidasi koordinat berada di dalam rentang Bounding Box resmi DIY.


In [ ]:
df_valid = df_osm_raw.dropna(subset=['latitude', 'longitude']).copy()
bbox = {'min_lat': -8.25, 'max_lat': -7.50, 'min_lon': 110.00, 'max_lon': 110.85}
mask_lat = (df_valid['latitude'] >= bbox['min_lat']) & (df_valid['latitude'] <= bbox['max_lat'])
mask_lon = (df_valid['longitude'] >= bbox['min_lon']) & (df_valid['longitude'] <= bbox['max_lon'])
df_valid = df_valid[mask_lat & mask_lon]
print('Data valid dalam Bounding Box:', len(df_valid))


# 04 Deduplikasi Spasial Dua Lapis
---
## Eliminasi Data Ganda
---
### Deduplikasi ID dan Titik Irisan Border Grid
---
Menghapus data kembar akibat irisan grid sampling spasial menggunakan verifikasi ID unik OSM dan koordinat presisi.


In [ ]:
n_awal = len(df_valid)
df_dedup_id = df_valid.drop_duplicates(subset=['osm_id', 'sektor', 'wilayah'])
df_clean = df_dedup_id.drop_duplicates(subset=['latitude', 'longitude', 'sektor', 'wilayah']).copy()
n_terbuang = n_awal - len(df_clean)
print('Total awal:', n_awal, '| Duplikat dibuang:', n_terbuang, '| Total bersih:', len(df_clean))


# 05 Rekapitulasi dan Penyimpanan Data Bersih
---
## Ringkasan Agregasi per Wilayah
---
### Ekspor File CSV Cleaned
---
Menghitung sebaran titik usaha per sektor di setiap kabupaten/kota, kemudian mengekspor dataset siap pakai.


In [ ]:
summary_sebaran = df_clean.groupby(['wilayah', 'sektor']).size().unstack(fill_value=0)
summary_sebaran


In [ ]:
output_osm_path = OUTPUT_DIR / 'kompetitor_per_wilayah.csv'
df_clean.to_csv(output_osm_path, index=False)
print('Dataset OSM bersih berhasil disimpan di:', output_osm_path)
